# Cross-encoder 3 (rescue step) for Business Entity Resolution (Ekdant.exe)
1. **Runtime → Change runtime type → T4 GPU**, then Save.
2. Click the **folder icon** on the left and upload ALL 6 files from `amazon-ml/colab_gpu3`: `xtrain_part00..03.parquet` and `xextra_part00..01.parquet`.
3. **Runtime → Run all.** It takes about 35–40 minutes. At the end, `xenc3_extra_scores.csv.gz` downloads automatically. Put it in the `amazon-ml` folder.

Model: `distilbert-base-multilingual-cased` (Apache-2.0), the same recipe as the v17 model. It only scores the rescue candidates (records that blocking gave no candidate).


In [ ]:
import glob, os, time, math, numpy as np, pandas as pd, torch
print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> T4 GPU'
n = {k: len(glob.glob(f'/content/{k}_part*.parquet')) for k in ('xtrain', 'xextra')}
print(n)
assert n == {'xtrain': 4, 'xextra': 2}, 'upload all 6 files first (folder icon on the left)'


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
torch.manual_seed(0); np.random.seed(0)
MODEL = 'distilbert-base-multilingual-cased'
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=1).cuda()
rd = lambda k: pd.concat([pd.read_parquet(f) for f in sorted(glob.glob(f'/content/{k}_part*.parquet'))], ignore_index=True)
tr = rd('xtrain'); print('train pairs', len(tr), 'positive share', tr.label.mean())
def enc(a, b):
    x = tok(list(a), list(b), truncation=True, max_length=128, padding=True, return_tensors='pt')
    return {k: v.cuda(non_blocking=True) for k, v in x.items()}

In [ ]:
BS, EPOCHS, LR = 128, 2, 5e-5
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
steps = EPOCHS * math.ceil(len(tr) / BS)
sch = get_linear_schedule_with_warmup(opt, int(0.05 * steps), steps)
scaler = torch.cuda.amp.GradScaler()
lossf = torch.nn.BCEWithLogitsLoss()
model.train()
t0 = time.time()
for ep in range(EPOCHS):
    idx = np.random.RandomState(ep).permutation(len(tr))
    for s, i in enumerate(range(0, len(idx), BS)):
        b = tr.iloc[idx[i:i + BS]]
        x = enc(b.a, b.b)
        y = torch.tensor(b.label.values, dtype=torch.float32).cuda()
        with torch.autocast('cuda', dtype=torch.float16):
            out = model(**x).logits.squeeze(-1)
        loss = lossf(out.float(), y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sch.step()
        if s % 500 == 0:
            print(f'epoch {ep} step {s}/{math.ceil(len(idx)/BS)} loss {loss.item():.4f} {time.time()-t0:.0f}s', flush=True)
print('training done', round(time.time() - t0), 's')

In [ ]:
model.eval()
def predict(df, bs=512):
    order = np.argsort((df.a.str.len() + df.b.str.len()).values)
    out = np.empty(len(df), dtype=np.float32)
    with torch.no_grad():
        for i in range(0, len(order), bs):
            j = order[i:i + bs]; b = df.iloc[j]
            with torch.autocast('cuda', dtype=torch.float16):
                out[j] = model(**enc(b.a, b.b)).logits.squeeze(-1).float().cpu().numpy()
    return out
xe = rd('xextra'); t0 = time.time(); xe['score'] = predict(xe); print('extra pairs', len(xe), round(time.time() - t0), 's')
xe[['rid', 's1', 'score', 'split']].to_csv('/content/xenc3_extra_scores.csv.gz', index=False)
print('written', len(xe), os.path.getsize('/content/xenc3_extra_scores.csv.gz') // 1_000_000, 'MB')


In [ ]:
from google.colab import files
files.download('/content/xenc3_extra_scores.csv.gz')   # put this file in your amazon-ml folder (if the download does not start: right-click the file in the Files panel -> Download)
